# Preprocessing Pipeline

Wraps the `run_pipeline` CLI in notebook form. Downloads raw IQ `.dat` files from S3,
extracts PSD, spectrogram, and IQ features, then saves them to disk.

The final cell exports test samples (`.npz` per drone/condition) for the Streamlit interface.

In [1]:
import logging
from pathlib import Path

from dronedetect.export_samples import export_test_samples
from dronedetect.pipeline import run_pipeline

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

## Configuration

Parameters below match CLI defaults. Adjust as needed (e.g. set `MAX_FILES = 5` for a quick test run).

In [2]:
FEATURES_TO_EXTRACT = {"psd", "spectrogram", "iq"}
CONCURRENCY = 2
OUTPUT_DIR = Path("../data/features")
UPLOAD = False
MAX_FILES = None  # Set to small number for testing (e.g., 5)

## Run Pipeline

Downloads raw `.dat` files from S3 one at a time, extracts features per segment, and saves results to `OUTPUT_DIR`. Supports resume via checkpoint file.

In [ ]:
run_pipeline(
    features_to_extract=FEATURES_TO_EXTRACT,
    concurrency=CONCURRENCY,
    output_dir=OUTPUT_DIR,
    upload=UPLOAD,
    max_files=MAX_FILES,
)

## Verify Output

Check that feature files were created and display their sizes.

In [ ]:
logger = logging.getLogger(__name__)

expected_files = [
    "psd_features.npz",
    "spectrogram_features.npy",
    "spectrogram_meta.npz",
    "iq_features.npz",
    "manifest.json",
]

for name in expected_files:
    path = OUTPUT_DIR / name
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        unit = "MB" if size_mb < 1000 else "GB"
        value = size_mb if size_mb < 1000 else size_mb / 1000
        print(f"{name}: {value:.1f} {unit}")
    else:
        print(f"{name}: MISSING")
        logger.warning("Expected file not found: %s", path)

## Export Test Samples

Creates per-condition `.npz` files from the test split for the Streamlit interface. Requires `split_indices.npz` (generated by `dronedetect.splitting`).

In [ ]:
SPLIT_PATH = OUTPUT_DIR.parent / "split_indices.npz"
SAMPLES_OUTPUT_DIR = Path("../data/test_samples")

In [5]:
export_test_samples(
    features_dir=OUTPUT_DIR,
    split_path=SPLIT_PATH,
    output_dir=SAMPLES_OUTPUT_DIR,
)

2026-05-07 20:21:54,631 [INFO] dronedetect.export_samples: Loaded split: 5622 test samples
2026-05-07 20:21:55,630 [INFO] dronedetect.export_samples: Loaded PSD features: ../data/features/psd_features.npz
2026-05-07 20:21:55,636 [INFO] dronedetect.export_samples: Loaded spectrogram features: ../data/features/spectrogram_features.npy
2026-05-07 20:22:24,599 [INFO] dronedetect.export_samples: Loaded IQ features: ../data/features/iq_features.npz
2026-05-07 20:22:24,600 [INFO] dronedetect.export_samples: Found 28 (drone, condition) combos in test set
2026-05-07 20:22:24,621 [INFO] dronedetect.export_samples: Exported ../interface/media/test_samples/BLUE/psd_BLUE_AIR.npz (122 segments, 364.7 KB)
2026-05-07 20:22:26,278 [INFO] dronedetect.export_samples: Exported ../interface/media/test_samples/BLUE/spectrogram_BLUE_AIR.npz (122 segments, 6768.3 KB)
2026-05-07 20:22:26,707 [INFO] dronedetect.export_samples: Exported ../interface/media/test_samples/BLUE/iq_BLUE_AIR.npz (122 segments, 5057.4 K